# QAF-Net · NB3 — Federated experiments (the main notebook)

**Accelerator: GPU P100. Internet: ON** (MobileNetV2 downloads its ImageNet weights).
~60–80 min per fold for all four methods, one seed.

**Attach:** the `qafnet nb1` notebook (+ Add Input → Notebooks). Nothing else.

Set `FOLD` and `SEED` at the top and change nothing else. Each teammate takes a different
`(FOLD, SEED)` pair — see `docs/03-kaggle-workflow.md` for the split.

| method | aggregates BN | prox `mu` | quant-aware `lam` |
|---|---|---|---|
| `fedavg`  | yes | – | – |
| `fedprox` | yes | 0.01 | – |
| `fedbn`   | **no** | – | – |
| `qafnet`  | **no** | 0.01 | 0.01 |

In [1]:
SMOKE_TEST   = False
FOLD         = "SARTAJ"       # held-out site: "CHENG" | "PMRAM" | "SARTAJ"
SEED         = 2
MODEL        = "mobilenet"   # "mobilenet" | "tinybtnet"
IMG, BATCH   = 128, 32
ROUNDS       = 2 if SMOKE_TEST else 25
LOCAL_EPOCHS = 1 if SMOKE_TEST else 2
LR           = 1e-3
MU, LAM      = 0.01, 0.01
# Primary setup after NB1's audit: 4 classes, every real site a client. Deduplication left SARTAJ
# with only 235 tumour images, so a 3-class, 3-site federation no longer exists; Br35H joins through
# the superset loss instead. Why: docs/01-research-design.md, section 3.
N_CLASSES     = 4               # glioma, meningioma, pituitary, notumor
CLASSES_USED  = (0, 1, 2, 3)
EXCLUDE       = ()              # every client site takes part
ALLOW_PARTIAL = True            # Br35H's "tumour, subtype unknown" images train via the superset loss
BN_MODES     = ("avg", "recalib")   # how an unseen site gets its BN statistics
RESULTS      = "/kaggle/working/results.csv"
RUN_ABLATION = False        # True: skip the four methods and run ONLY the lambda sweep

METHODS = {
    "fedavg":  dict(mu=0.0, lam=0.0, sln=False),
    "fedprox": dict(mu=MU,  lam=0.0, sln=False),
    "fedbn":   dict(mu=0.0, lam=0.0, sln=True),
    "qafnet":  dict(mu=MU,  lam=LAM, sln=True),
}

## Shared core

_Inlined from `src/fedtiny/core.py` by `tools/build_notebooks.py`. Edit the source file, not this cell._

In [2]:
# QAF-Net core — shared by NB2/NB3/NB4.
# tools/build_notebooks.py inlines this file wherever a notebook has `# %%INCLUDE core`,
# so this is the single source of truth. Edit here, never in a notebook copy.

import os, io, json, time, math, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)

CLASSES = ["glioma", "meningioma", "pituitary", "notumor"]
AUTOTUNE = tf.data.AUTOTUNE


# ---------------------------------------------------------------- reproducibility
def set_seed(s):
    os.environ["PYTHONHASHSEED"] = str(s)
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)


# ---------------------------------------------------------------- data
def load_prepared(npz_path):
    """Load NB1's output. Returns a dict of parallel arrays."""
    z = np.load(npz_path, allow_pickle=True)
    d = dict(images=z["images"], labels=z["labels"].astype(np.int32),
             partial=z["partial"].astype(np.int32), site=z["site"].astype(str),
             group=z["group"].astype(str), store=int(z["store"][0]))
    print(f"loaded {len(d['images'])} images @ {d['store']}px  sites={sorted(set(d['site']))}")
    if "smoke_test" in z.files and bool(z["smoke_test"][0]):
        print("!! WARNING: this is NB1's SMOKE_TEST output (~200 images per site). Fine for a "
              "smoke test, but re-commit NB1 with SMOKE_TEST = False before any real run.")
    if "dedup" in z.files and not bool(z["dedup"][0]):
        print("!! NOTE: NB1 ran with DEDUP = False, so cross-site duplicates are INCLUDED. "
              "Use this only for the C1 leakage ablation.")
    return d


def label_sets(labels, partial, n_classes):
    """Multi-hot allowed-class matrix. Singleton normally; {glioma,men,pit} for partial labels."""
    Y = np.zeros((len(labels), n_classes), np.float32)
    for i, (l, p) in enumerate(zip(labels, partial)):
        if p:
            Y[i, :3] = 1.0                      # tumour of unspecified subtype
        else:
            Y[i, l] = 1.0
    return Y


def loso_split(data, held_out_site, classes_used=(0, 1, 2), val_frac=0.15, seed=0,
               exclude_sites=(), allow_partial=False):
    """Leave-One-Site-Out. Returns (clients, test_idx).

    `clients` is a list of (site_name, train_idx, val_idx). Splits are grouped by patient where
    patient IDs exist, so slices from one patient never straddle train and val.

    `allow_partial=True` admits set-valued labels (Br35H's tumour-of-unknown-subtype images) to
    training through the superset loss, as the primary 4-class setup does. With False, those
    clients are left out entirely.

    Partial-label samples never enter a val or test split under any setting: you cannot score a
    prediction against a label you do not know.
    """
    rng = np.random.RandomState(seed)
    keep_cls = set(classes_used)
    labelled = np.array([l in keep_cls for l in data["labels"]]) & (data["partial"] == 0)
    usable = labelled | ((data["partial"] == 1) if allow_partial else False)

    test_idx = np.where(labelled & (data["site"] == held_out_site))[0]

    clients = []
    for s in sorted(set(data["site"])):
        if s == held_out_site or s in exclude_sites:
            continue
        idx = np.where(usable & (data["site"] == s))[0]
        if len(idx) < 50:
            print(f"  (skipping {s}: only {len(idx)} usable images)")
            continue
        groups = data["group"][idx]
        uniq = np.unique(groups); rng.shuffle(uniq)
        n_val = max(1, int(len(uniq) * val_frac))
        val_g = set(uniq[:n_val])
        is_val = np.array([g in val_g for g in groups]) & (data["partial"][idx] == 0)
        clients.append((s, idx[~is_val], idx[is_val]))

    print(f"LOSO held-out={held_out_site}  test={len(test_idx)}")
    for s, tr, va in clients:
        npart = int(data["partial"][tr].sum())
        print(f"  client {s:8s} train={len(tr):5d} (partial={npart:4d}) val={len(va):4d} "
              f"groups={len(set(data['group'][tr]))}")
    return clients, test_idx


def make_ds(data, idx, n_classes, img=128, batch=32, train=False, shuffle=True, seed=0):
    """tf.data pipeline. Stored images are (S,S) uint8 grayscale; models want (img,img,3) float."""
    X = data["images"][idx]
    Y = label_sets(data["labels"][idx], data["partial"][idx], n_classes)
    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    if train and shuffle:
        ds = ds.shuffle(min(len(idx), 4096), seed=seed, reshuffle_each_iteration=True)

    S = data["store"]

    def prep(x, y):
        x = tf.cast(x, tf.float32)[..., None]
        if train:
            x = tf.image.random_crop(x, [img, img, 1])        # translation jitter from S -> img
            # PMRAM slices are stored rotated by +/-90 degrees relative to the other sites (NB1's
            # sample grid). Random quarter turns plus the flip below cover all eight orientations,
            # so no site's export convention can become a shortcut.
            x = tf.image.rot90(x, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
            x = tf.image.random_flip_left_right(x)
            x = tf.image.random_brightness(x, 20.0)
            x = tf.image.random_contrast(x, 0.85, 1.15)
        else:
            off = (S - img) // 2
            x = tf.image.crop_to_bounding_box(x, off, off, img, img)
        x = tf.clip_by_value(x, 0.0, 255.0) / 127.5 - 1.0      # MobileNetV2 input range
        x = tf.repeat(x, 3, axis=-1)
        return x, y

    return ds.map(prep, num_parallel_calls=AUTOTUNE).batch(batch).prefetch(AUTOTUNE)


# ---------------------------------------------------------------- models
def build_mobilenet(n_classes, img=128, alpha=0.35, dropout=0.3, pretrained=True):
    """Built on `base.output` rather than `base(inp)` so the graph stays FLAT.

    Wrapping the backbone as a nested submodel would put its conv layers in a separate graph,
    which breaks Grad-CAM (`keras.Model(model.inputs, [conv.output, model.output])` raises a
    disconnected-graph error) and complicates weight bookkeeping for no benefit.
    """
    try:
        base = keras.applications.MobileNetV2(input_shape=(img, img, 3), alpha=alpha,
                                              include_top=False,
                                              weights="imagenet" if pretrained else None)
    except Exception as e:
        # Keras fetches the ImageNet weights from the internet on first use.
        if pretrained and any(s in str(e) for s in ("URL fetch", "resolution", "urlopen",
                                                     "unreachable", "Connection", "HTTP Error")):
            raise RuntimeError(
                "MobileNetV2 could not download its ImageNet weights. Turn Internet ON for this "
                "notebook (right panel -> Settings -> Internet) and run again.") from e
        raise
    base.trainable = True
    x = keras.layers.GlobalAveragePooling2D()(base.output)
    x = keras.layers.Dropout(dropout)(x)
    out = keras.layers.Dense(n_classes, name="head")(x)        # logits, no softmax
    return keras.Model(base.input, out, name=f"mnv2_a{alpha}_{img}")


def build_tinybtnet(n_classes, img=128, w=16, dropout=0.3):
    """~0.18 M params. Sub-200 KB once INT8-quantized — the MCU regime."""
    def blk(x, f, s=1):
        x = keras.layers.SeparableConv2D(f, 3, strides=s, padding="same", use_bias=False)(x)
        x = keras.layers.BatchNormalization()(x)
        return keras.layers.ReLU(6.0)(x)

    inp = keras.Input((img, img, 3))
    x = keras.layers.Conv2D(w, 3, strides=2, padding="same", use_bias=False)(inp)
    x = keras.layers.BatchNormalization()(x); x = keras.layers.ReLU(6.0)(x)
    for f, s in [(w*2, 2), (w*2, 1), (w*4, 2), (w*4, 1), (w*8, 2), (w*8, 1), (w*12, 2)]:
        x = blk(x, f, s)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(dropout)(x)
    out = keras.layers.Dense(n_classes, name="head")(x)
    return keras.Model(inp, out, name=f"tinybtnet_w{w}_{img}")


def build_model(name, n_classes, img=128, pretrained=True):
    """`pretrained=False` skips the ImageNet download. Use it when a checkpoint overwrites every
    weight anyway (NB4), so that notebook runs with Internet off."""
    return {"mobilenet": lambda: build_mobilenet(n_classes, img, pretrained=pretrained),
            "tinybtnet": lambda: build_tinybtnet(n_classes, img)}[name]()


# ---------------------------------------------------------------- variable bookkeeping
def _iter_layers(m):
    for l in m.layers:
        if hasattr(l, "layers"):
            yield from _iter_layers(l)
        else:
            yield l


def bn_indices(model):
    """Positions in model.get_weights() belonging to BatchNorm.

    Matched by object identity, not by name or index arithmetic — nested submodels (MobileNetV2)
    make positional guessing unreliable.
    """
    ids = set()
    for l in _iter_layers(model):
        if isinstance(l, keras.layers.BatchNormalization):
            ids.update(id(v) for v in l.weights)
    return [i for i, v in enumerate(model.weights) if id(v) in ids]


def quant_targets(model):
    """{id(kernel_variable): reduction_axes} for conv/dense kernels.

    Axes match TFLite's per-output-channel scheme: last axis for conv/dense, axis 2 for depthwise
    (whose output channels live on the input-channel dimension).
    """
    out = {}
    for l in _iter_layers(model):
        k = getattr(l, "kernel", None)
        if k is None:
            k = getattr(l, "depthwise_kernel", None)
        if k is None or len(k.shape) < 2:
            continue
        if isinstance(l, keras.layers.DepthwiseConv2D) or k is getattr(l, "depthwise_kernel", None):
            axes = [0, 1, 3]                             # depthwise: channels live on axis 2
        else:
            axes = list(range(len(k.shape) - 1))
        out[id(k)] = axes
        pw = getattr(l, "pointwise_kernel", None)          # SeparableConv2D has two kernels
        if pw is not None:
            out[id(pw)] = list(range(len(pw.shape) - 1))
    return out


def fake_quant(w, axes, bits=8):
    """Per-channel symmetric INT8 round-trip. Differentiable target is stop_gradiented by caller."""
    qmax = float(2 ** (bits - 1) - 1)
    s = tf.reduce_max(tf.abs(w), axis=axes, keepdims=True) / qmax
    s = tf.maximum(s, 1e-12)
    return tf.round(w / s) * s


# ---------------------------------------------------------------- losses
def superset_ce(y_sets, logits):
    """-log sum_{c in S} p_c. Reduces to standard cross-entropy when |S| = 1."""
    logp = tf.nn.log_softmax(logits)
    masked = tf.where(y_sets > 0, logp, tf.fill(tf.shape(logp), -1e9))
    return -tf.reduce_logsumexp(masked, axis=-1)


# ---------------------------------------------------------------- local training
class LocalTrainer:
    """One client's local optimizer.

        L = superset_CE + (mu/2)||w - w_global||^2 + lam * mean_j (w_j - Q(w_j))^2

    mu  > 0  -> FedProx drift control
    lam > 0  -> QAF-Net quantization-aware term (contribution C2)

    The `tf.function` is traced **once** and reused across every client and every round; anchors
    live in `tf.Variable`s that get reassigned. Rebuilding it per call would retrace the graph
    hundreds of times per experiment and dominate the runtime.
    """

    def __init__(self, model, lr, mu=0.0, lam=0.0):
        self.model, self.mu, self.lam = model, mu, lam
        self.opt = keras.optimizers.Adam(lr)
        self.tvars = model.trainable_variables
        self.by_id = {id(v): i for i, v in enumerate(model.weights)}

        qmap = quant_targets(model) if lam > 0 else {}
        self.qaxes = [qmap.get(id(v)) for v in self.tvars]
        self.anchors = ([tf.Variable(tf.zeros_like(v), trainable=False) for v in self.tvars]
                        if mu > 0 else None)
        try:
            self.opt.build(self.tvars)          # Keras 3; older versions build lazily
        except Exception:
            pass
        self._step = tf.function(self._step_impl, reduce_retracing=True)

    def _step_impl(self, x, y):
        with tf.GradientTape() as tape:
            logits = self.model(x, training=True)
            loss = tf.reduce_mean(superset_ce(y, logits))
            if self.anchors is not None:
                loss += (self.mu / 2.0) * tf.add_n(
                    [tf.reduce_sum(tf.square(v - a)) for v, a in zip(self.tvars, self.anchors)])
            if self.lam > 0:
                terms = [tf.reduce_mean(tf.square(v - tf.stop_gradient(fake_quant(v, ax))))
                         for v, ax in zip(self.tvars, self.qaxes) if ax is not None]
                if terms:
                    loss += self.lam * tf.add_n(terms)
        self.opt.apply_gradients(zip(tape.gradient(loss, self.tvars), self.tvars))
        return loss

    def set_anchor(self, global_w):
        if self.anchors is None:
            return
        for a, v in zip(self.anchors, self.tvars):
            a.assign(global_w[self.by_id[id(v)]])

    def reset_optimizer(self):
        """FL clients are stateless — Adam moments must not carry from one client to the next."""
        # Reset ONLY Adam's moment slots. Keras 3 keeps the learning rate itself among
        # `optimizer.variables`; zeroing those set lr = 0, so nothing trained.
        slots = []
        for attr in ("_momentums", "_velocities", "_velocity_hats"):
            slots += list(getattr(self.opt, attr, None) or [])
        if not slots:
            vs = getattr(self.opt, "variables", None)
            vs = vs() if callable(vs) else (vs or [])
            slots = [v for v in vs
                     if not any(k in v.name for k in ("learning_rate", "iteration"))]
        for v in slots:
            v.assign(tf.zeros_like(v))
        if hasattr(self.opt, "iterations"):
            self.opt.iterations.assign(0)
        lr = float(np.asarray(self.opt.learning_rate))
        assert lr > 0, f"learning rate is {lr} after optimizer reset"

    def run(self, ds, epochs, verbose=0):
        hist = []
        for e in range(epochs):
            tot = n = 0.0
            for x, y in ds:
                tot += float(self._step(x, y)); n += 1
            hist.append(tot / max(n, 1))
            if verbose:
                print(f"    epoch {e+1}/{epochs} loss={hist[-1]:.4f}")
        return hist


# ---------------------------------------------------------------- aggregation
def fedavg(client_ws, sizes, skip=()):
    """Size-weighted parameter average. Indices in `skip` keep client-0's values untouched
    (used for site-local BN — those are restored per client anyway)."""
    w = np.asarray(sizes, np.float64); w /= w.sum()
    skip = set(skip)
    out = []
    for j in range(len(client_ws[0])):
        if j in skip:
            out.append(np.array(client_ws[0][j]))
        else:
            out.append(sum(wi * np.asarray(cw[j]) for wi, cw in zip(w, client_ws)))
    return out


def bn_average(client_ws, sizes, bn_idx):
    """Size-weighted average of BN parameters only — the default global BN for an unseen site."""
    w = np.asarray(sizes, np.float64); w /= w.sum()
    return {j: sum(wi * np.asarray(cw[j]) for wi, cw in zip(w, client_ws)) for j in bn_idx}


def bn_recalibrate(model, ds, steps=100):
    """Re-estimate BN statistics on the TARGET site using unlabeled forward passes only.

    No labels, no gradients, no data leaves the site — this is the deployment-time calibration
    step behind contribution C3.
    """
    saved = [(l, float(l.momentum)) for l in _iter_layers(model)
             if isinstance(l, keras.layers.BatchNormalization)]
    for l, _ in saved:
        l.momentum = 0.9
    n = 0
    for x, _ in ds:
        model(x, training=True)                 # updates moving_mean / moving_variance only
        n += 1
        if n >= steps:
            break
    for l, m in saved:
        l.momentum = m
    return n


# ---------------------------------------------------------------- evaluation
def _metrics(yt, prob, n_classes):
    """Scores for one evaluation: `yt` true class ids, `prob` softmax rows.

    macro-F1 averages over the classes present in the test set, so a 4-class model tested on a
    tumour-only site is not charged an undefined F1 for "notumor"; predicting "notumor" there still
    costs recall on the true classes. `acc_sub` scores tumour images on the three tumour logits
    alone (subtype accuracy). `tumor_missed` is the share of tumour images called "notumor".
    """
    yt = np.asarray(yt)
    yp = prob.argmax(1)
    present = sorted(set(yt.tolist()))
    m = dict(acc=float(accuracy_score(yt, yp)),
             macro_f1=float(f1_score(yt, yp, labels=present, average="macro", zero_division=0)))
    try:
        m["auc"] = float(roc_auc_score(np.eye(n_classes)[yt][:, present], prob[:, present],
                                       average="macro", multi_class="ovr"))
    except Exception:
        m["auc"] = float("nan")
    tumour = yt < 3
    if tumour.any():
        m["acc_sub"] = float((prob[tumour, :3].argmax(1) == yt[tumour]).mean())
        m["tumor_missed"] = float((yp[tumour] == 3).mean()) if n_classes > 3 else 0.0
    else:
        m["acc_sub"] = m["tumor_missed"] = float("nan")
    m["cm"] = confusion_matrix(yt, yp, labels=list(range(n_classes))).tolist()
    return m


def evaluate(model, ds, n_classes, name=""):
    logits, ys = [], []
    for x, y in ds:
        logits.append(model(x, training=False).numpy()); ys.append(y.numpy())
    logits = np.concatenate(logits); Y = np.concatenate(ys)
    prob = tf.nn.softmax(logits).numpy()
    yt = Y.argmax(1)                                    # test sets are always singleton-labelled
    m = _metrics(yt, prob, n_classes)
    m["report"] = classification_report(yt, prob.argmax(1), labels=list(range(n_classes)),
                                        target_names=CLASSES[:n_classes],
                                        output_dict=True, zero_division=0)
    if name:
        print(f"  {name:26s} acc={m['acc']:.4f} subtype={m['acc_sub']:.4f} "
              f"macroF1={m['macro_f1']:.4f} auc={m['auc']:.4f}")
    return m


# ---------------------------------------------------------------- TinyML export
def to_tflite_int8(model, rep_ds, n_batches=100, workdir="/kaggle/working/_export"):
    """Full-integer INT8 conversion — int8 in, int8 out, MCU-deployable."""
    def rep_gen():
        for i, (x, _) in enumerate(rep_ds):
            if i >= n_batches:
                break
            yield [tf.cast(x, tf.float32)]

    def _cfg(c):
        c.optimizations = [tf.lite.Optimize.DEFAULT]
        c.representative_dataset = rep_gen
        c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        c.inference_input_type = tf.int8
        c.inference_output_type = tf.int8
        return c

    try:
        return _cfg(tf.lite.TFLiteConverter.from_keras_model(model)).convert()
    except Exception as e:
        # Keras 3 sometimes needs the SavedModel route.
        print(f"  from_keras_model failed ({type(e).__name__}); using SavedModel export")
        os.makedirs(workdir, exist_ok=True)
        p = os.path.join(workdir, model.name)
        if hasattr(model, "export"):
            model.export(p)                     # Keras 3
        else:
            tf.saved_model.save(model, p)       # Keras 2
        return _cfg(tf.lite.TFLiteConverter.from_saved_model(p)).convert()


def tflite_interpreter(tflite_bytes):
    """TFLite interpreter from LiteRT when installed, otherwise tf.lite.

    tf.lite.Interpreter is deprecated and scheduled for removal from TensorFlow. It still exists in
    TF 2.20, which Kaggle runs today; preferring ai_edge_litert keeps these notebooks working when
    Kaggle's image moves past it.
    """
    try:
        from ai_edge_litert.interpreter import Interpreter
    except ImportError:
        Interpreter = getattr(getattr(tf, "lite", None), "Interpreter", None)
        if Interpreter is None:
            raise ImportError("This TensorFlow has no tf.lite.Interpreter. Turn Settings -> Internet "
                              "on and run:  !pip install ai-edge-litert")
    return Interpreter(model_content=tflite_bytes)


def eval_tflite(tflite_bytes, ds, n_classes, name=""):
    """Evaluate the quantized model, honouring the interpreter's own input/output quant params."""
    it = tflite_interpreter(tflite_bytes)
    it.allocate_tensors()
    ind, outd = it.get_input_details()[0], it.get_output_details()[0]
    is_, iz = ind["quantization"]
    os_, oz = outd["quantization"]
    # A zero scale means the tensor came back unquantized; treat it as identity rather than
    # dividing by zero and silently producing garbage predictions.
    if is_ == 0: is_, iz = 1.0, 0
    if os_ == 0: os_, oz = 1.0, 0

    yt, yp, probs, lat = [], [], [], []
    for x, y in ds:
        for xi, yi in zip(x.numpy(), y.numpy()):
            q = np.clip(np.round(xi / is_ + iz), -128, 127).astype(ind["dtype"])[None]
            it.set_tensor(ind["index"], q)
            t0 = time.perf_counter(); it.invoke(); lat.append((time.perf_counter() - t0) * 1e3)
            o = it.get_tensor(outd["index"])[0].astype(np.float32)
            logits = (o - oz) * os_
            p = np.exp(logits - logits.max()); p /= p.sum()
            probs.append(p); yp.append(int(p.argmax())); yt.append(int(yi.argmax()))

    m = _metrics(np.array(yt), np.stack(probs), n_classes)
    m.update(latency_ms_mean=float(np.mean(lat)), latency_ms_p95=float(np.percentile(lat, 95)),
             size_kb=len(tflite_bytes) / 1024.0)
    if name:
        print(f"  {name:26s} acc={m['acc']:.4f} subtype={m['acc_sub']:.4f} macroF1={m['macro_f1']:.4f} "
              f"size={m['size_kb']:.0f}KB lat={m['latency_ms_mean']:.2f}ms")
    return m


def model_cost(model):
    """Parameter count and multiply-accumulate estimate for conv/dense layers."""
    macs = 0
    for l in _iter_layers(model):
        try:
            osh = l.output.shape
        except Exception:
            continue
        if isinstance(l, keras.layers.DepthwiseConv2D):
            dk = getattr(l, "depthwise_kernel", None)
            if dk is None:
                dk = l.kernel                            # Keras 3 renamed depthwise_kernel to kernel
            kh, kw, cin, mult = dk.shape
            macs += int(osh[1]) * int(osh[2]) * int(kh) * int(kw) * int(cin) * int(mult)
        elif isinstance(l, keras.layers.SeparableConv2D):
            kh, kw, cin, mult = l.depthwise_kernel.shape
            macs += int(osh[1]) * int(osh[2]) * (int(kh)*int(kw)*int(cin)*int(mult)
                                                 + int(cin)*int(mult)*int(osh[3]))
        elif isinstance(l, keras.layers.Conv2D):
            kh, kw, cin, cout = l.kernel.shape
            macs += int(osh[1]) * int(osh[2]) * int(kh) * int(kw) * int(cin) * int(cout)
        elif isinstance(l, keras.layers.Dense):
            macs += int(l.kernel.shape[0]) * int(l.kernel.shape[1])
    return dict(params=int(model.count_params()), macs=int(macs))


# ---------------------------------------------------------------- results log
def log_result(path, **row):
    """Add one run to results.csv under a single consistent header.

    Notebooks log different fields (local-only runs have no internal-validation numbers), so rows
    are merged by column name. Appending blindly would put values under the wrong headers.
    """
    row = {k: (json.dumps(v) if isinstance(v, (list, dict)) else v) for k, v in row.items()}
    df = pd.DataFrame([row])
    if os.path.exists(path):
        df = pd.concat([pd.read_csv(path), df], ignore_index=True, sort=False)
    df.to_csv(path, index=False)
    return df


In [3]:
import glob, copy
set_seed(SEED)
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

npz = (glob.glob("/kaggle/input/**/qafnet_data.npz", recursive=True) or [None])[0]
assert npz, ("qafnet_data.npz not found. Commit NB1 first, then + Add Input -> Notebooks -> "
             "qafnet nb1. Uploading raw data will not fix this.")
data = load_prepared(npz)

clients, test_idx = loso_split(data, FOLD, CLASSES_USED, seed=SEED,
                              exclude_sites=EXCLUDE, allow_partial=ALLOW_PARTIAL)
assert len(clients) >= 2, f"need >=2 training sites, got {len(clients)} — check NB1 survival rates"

client_ds = [(s, make_ds(data, tr, N_CLASSES, IMG, BATCH, train=True, seed=SEED),
                 make_ds(data, va, N_CLASSES, IMG, BATCH), len(tr))
             for s, tr, va in clients]
val_all = make_ds(data, np.concatenate([c[2] for c in clients]), N_CLASSES, IMG, BATCH)
test_ds = make_ds(data, test_idx, N_CLASSES, IMG, BATCH)
rep_ds  = make_ds(data, np.concatenate([c[1] for c in clients]), N_CLASSES, IMG, BATCH, train=True, seed=SEED)

TF 2.20.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
loaded 11528 images @ 144px  sites=[np.str_('BR35H'), np.str_('CHENG'), np.str_('PMRAM'), np.str_('SARTAJ')]
LOSO held-out=SARTAJ  test=544
  client BR35H    train= 1803 (partial=1275) val= 105 groups=1803
  client CHENG    train= 2657 (partial=   0) val= 407 groups=199
  client PMRAM    train= 5111 (partial=   0) val= 901 groups=5111


I0000 00:00:1790007009.107100      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790007009.113336      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


## The federated loop

Standard FedAvg skeleton with two switches:

* **`sln`** — site-local normalization. BatchNorm parameters and running statistics are excluded
  from aggregation and restored per client each round, so each site keeps its own scanner
  intensity profile instead of averaging it away.
* **`lam`** — the quantization-aware term inside `LocalTrainer`, which pulls every client's
  kernels toward an INT8 grid *during* training so that averaging happens between
  already-quantization-friendly weight sets.

In [4]:
def run_federated(method, cfg, seed=SEED):
    set_seed(seed)
    model = build_model(MODEL, N_CLASSES, IMG)
    cost = model_cost(model)
    bn_idx = bn_indices(model)
    print(f"\n{'='*70}\n{method.upper()} | held-out={FOLD} seed={seed} | {cost} | "
          f"{len(bn_idx)} BN tensors\n{'='*70}")

    trainer = LocalTrainer(model, LR, mu=cfg["mu"], lam=cfg["lam"])
    global_w = model.get_weights()
    local_bn = {s: [np.array(global_w[j]) for j in bn_idx] for s, _, _, _ in client_ds}
    sizes = [n for _, _, _, n in client_ds]
    history = []

    for r in range(ROUNDS):
        t0 = time.time(); client_ws, losses = [], []
        for (site, d_tr, _, n) in client_ds:
            w = [np.array(x) for x in global_w]
            if cfg["sln"]:                                   # restore this site's own BN
                for k, j in enumerate(bn_idx):
                    w[j] = local_bn[site][k]
            model.set_weights(w)
            trainer.reset_optimizer()
            trainer.set_anchor(global_w)
            losses.append(trainer.run(d_tr, LOCAL_EPOCHS)[-1])
            cw = model.get_weights()
            if cfg["sln"]:
                local_bn[site] = [np.array(cw[j]) for j in bn_idx]
            client_ws.append(cw)

        # BN tensors are skipped by the aggregator when SLN is on; they stay per-client.
        global_w = fedavg(client_ws, sizes, skip=bn_idx if cfg["sln"] else ())
        if cfg["sln"]:
            for j, v in bn_average(client_ws, sizes, bn_idx).items():
                global_w[j] = v                              # global BN = size-weighted client avg

        model.set_weights(global_w)
        va = evaluate(model, val_all, N_CLASSES)
        history.append(dict(round=r + 1, loss=float(np.mean(losses)), val_acc=va["acc"]))
        print(f"  round {r+1:2d}/{ROUNDS}  loss={np.mean(losses):.4f}  "
              f"val_acc={va['acc']:.4f}  ({time.time()-t0:.0f}s)")

    # ---- evaluate the deployable global model on the completely unseen site ----
    out = {}
    for bn_mode in BN_MODES:
        model.set_weights(global_w)
        if bn_mode == "recalib":
            # unlabeled forward passes on the target site only — no labels, no gradients
            n = bn_recalibrate(model, test_ds, steps=10 if SMOKE_TEST else 100)
            print(f"  BN recalibrated on {n} unlabeled target batches")

        fp_in  = evaluate(model, val_all, N_CLASSES, f"[{bn_mode}] FP32 internal-val")
        fp_out = evaluate(model, test_ds, N_CLASSES, f"[{bn_mode}] FP32 held-out {FOLD}")

        tfl = to_tflite_int8(model, rep_ds, n_batches=20 if SMOKE_TEST else 100)
        q_in  = eval_tflite(tfl, val_all, N_CLASSES, f"[{bn_mode}] INT8 internal-val")
        q_out = eval_tflite(tfl, test_ds, N_CLASSES, f"[{bn_mode}] INT8 held-out {FOLD}")

        dq = fp_out["acc"] - q_out["acc"]
        dq_sub = fp_out["acc_sub"] - q_out["acc_sub"]
        print(f"  >> Δ(FP32→INT8) held-out: accuracy {dq:+.4f}, subtype {dq_sub:+.4f}   <-- the headline numbers")

        if bn_mode == BN_MODES[0]:
            stem = f"/kaggle/working/{method}_{FOLD}_s{seed}_{MODEL}"
            open(f"{stem}.tflite", "wb").write(tfl)
            model.save_weights(f"{stem}.weights.h5")   # NB4 needs these for the weight-distribution
            json.dump(history, open(f"{stem}_history.json", "w"))   # and the Grad-CAM figure

        log_result(RESULTS, method=method, bn_mode=bn_mode, fold=FOLD, seed=seed, model=MODEL,
                   n_classes=N_CLASSES, rounds=ROUNDS, local_epochs=LOCAL_EPOCHS,
                   mu=cfg["mu"], lam=cfg["lam"], sln=int(cfg["sln"]),
                   n_clients=len(client_ds), n_train=int(sum(sizes)),
                   acc_fp32_in=fp_in["acc"], f1_fp32_in=fp_in["macro_f1"],
                   acc_fp32_out=fp_out["acc"], f1_fp32_out=fp_out["macro_f1"],
                   auc_fp32_out=fp_out["auc"],
                   acc_int8_in=q_in["acc"], acc_int8_out=q_out["acc"],
                   f1_int8_out=q_out["macro_f1"],
                   dq_in=fp_in["acc"]-q_in["acc"], dq_out=dq,
                   acc_sub_fp32_out=fp_out["acc_sub"], acc_sub_int8_out=q_out["acc_sub"],
                   dq_sub_out=dq_sub, tumor_missed_int8_out=q_out["tumor_missed"],
                   size_kb=q_out["size_kb"], latency_ms=q_out["latency_ms_mean"],
                   latency_p95=q_out["latency_ms_p95"],
                   cm_fp32_out=fp_out["cm"], cm_int8_out=q_out["cm"],
                   history=history, **cost)
        out[bn_mode] = (fp_out, q_out)
    return out

In [5]:
if not RUN_ABLATION:
    for name, cfg in METHODS.items():
        run_federated(name, cfg)

2019640/2019640 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

FEDAVG | held-out=SARTAJ seed=2 | {'params': 415332, 'macs': 18945792} | 208 BN tensors
  round  1/25  loss=0.2370  val_acc=0.4126  (67s)
  round  2/25  loss=0.2035  val_acc=0.3383  (42s)
  round  3/25  loss=0.1856  val_acc=0.3659  (42s)
  round  4/25  loss=0.1806  val_acc=0.4989  (41s)
  round  5/25  loss=0.1495  val_acc=0.6299  (42s)
  round  6/25  loss=0.1336  val_acc=0.7084  (42s)
  round  7/25  loss=0.1360  val_acc=0.6688  (42s)
  round  8/25  loss=0.1436  val_acc=0.5655  (42s)
  round  9/25  loss=0.1579  val_acc=0.8620  (42s)
  round 10/25  loss=0.1182  val_acc=0.8634  (42s)
  round 11/25  loss=0.0998  val_acc=0.8195  (42s)
  round 12/25  loss=0.1359  val_acc=0.8266  (42s)
  round 13/25  loss=0.1221  val_acc=0.8669  (41s)
  round 14/25  loss=0.1254  val_acc=0.7728  (41s)
  round 15/25  loss=0.0980  val_acc=0.8075  (42s)
  round 16/25  loss=0.1168  val_acc=0.7742  (42s)
  round 17/25  loss=0.1026  val_acc=0.8217  (41s)
  round 18/2

INFO:tensorflow:Assets written to: /tmp/tmpeevevne0/assets


Saved artifact at '/tmp/tmpeevevne0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  133975637915984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133975637915792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133975637916944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133975637916176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133975637915024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133975637914832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133975637916752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133975637915408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133975637915216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133975637916560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13397563791636

W0000 00:00:1790008097.381996      23 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1790008097.382028      23 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1790008097.513058      23 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


  [avg] INT8 internal-val    acc=0.8896 subtype=0.9610 macroF1=0.6937 size=606KB lat=1.28ms
  [avg] INT8 held-out SARTAJ acc=0.2537 subtype=0.5872 macroF1=0.2376 size=606KB lat=1.27ms
  >> Δ(FP32→INT8) held-out: accuracy +0.0092, subtype +0.0213   <-- the headline numbers
  BN recalibrated on 17 unlabeled target batches
  [recalib] FP32 internal-val acc=0.8981 subtype=0.9702 macroF1=0.7006 auc=0.9447
  [recalib] FP32 held-out SARTAJ acc=0.2665 subtype=0.6170 macroF1=0.2617 auc=0.7266
INFO:tensorflow:Assets written to: /tmp/tmpnkx6jxmk/assets


...

  [avg] INT8 internal-val    acc=0.8401 subtype=0.9067 macroF1=0.6599 size=606KB lat=1.31ms
  [avg] INT8 held-out SARTAJ acc=0.2849 subtype=0.6596 macroF1=0.2318 size=606KB lat=1.27ms
  >> Δ(FP32→INT8) held-out: accuracy +0.0000, subtype -0.0043   <-- the headline numbers
  BN recalibrated on 17 unlabeled target batches
  [recalib] FP32 internal-val acc=0.9052 subtype=0.9633 macroF1=0.7771 auc=0.9841
  [recalib] FP32 held-out SARTAJ acc=0.2978 subtype=0.6638 macroF1=0.2840 auc=0.7655
INFO:tensorflow:Assets written to: /tmp/tmpbq3b2u47/assets


INFO:tensorflow:Assets written to: /tmp/tmpbq3b2u47/assets


Saved artifact at '/tmp/tmpbq3b2u47'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor_471')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  133970263026064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133970263027600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133970263027408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133970263026448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133970263028176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133970263026640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133970263027792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133970263027984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133970263026832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133970263029136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1339702630

W0000 00:00:1790012077.691766      23 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1790012077.691798      23 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


  [recalib] INT8 internal-val acc=0.9052 subtype=0.9641 macroF1=0.7738 size=606KB lat=1.28ms
  [recalib] INT8 held-out SARTAJ acc=0.2941 subtype=0.6638 macroF1=0.2801 size=606KB lat=1.32ms
  >> Δ(FP32→INT8) held-out: accuracy +0.0037, subtype +0.0000   <-- the headline numbers


## The lambda ablation (run once, on one fold)
Sweeps the quantization-aware weight to show the accuracy/quantizability trade-off.
`lam=0` is exactly FedProx + SLN, so the sweep contains its own control.

In [6]:
if RUN_ABLATION:
    for lam in [0.0, 1e-3, 1e-2, 1e-1, 1.0]:
        run_federated(f"qafnet_lam{lam}", dict(mu=MU, lam=lam, sln=True))

In [7]:
df = pd.read_csv(RESULTS)
print(df[["method", "bn_mode", "fold", "acc_fp32_out", "acc_int8_out", "dq_out",
          "acc_sub_int8_out", "dq_sub_out", "size_kb", "latency_ms"]].to_string(index=False))

 method bn_mode   fold  acc_fp32_out  acc_int8_out    dq_out  acc_sub_int8_out  dq_sub_out    size_kb  latency_ms
 fedavg     avg SARTAJ      0.262868      0.253676  0.009191          0.587234    0.021277 606.296875    1.268428
 fedavg recalib SARTAJ      0.266544      0.233456  0.033088          0.540426    0.076596 606.296875    1.292478
fedprox     avg SARTAJ      0.290441      0.294118 -0.003676          0.629787   -0.004255 606.312500    1.304505
fedprox recalib SARTAJ      0.334559      0.306985  0.027574          0.646809    0.042553 606.312500    1.273773
  fedbn     avg SARTAJ      0.268382      0.257353  0.011029          0.595745    0.025532 606.312500    1.342683
  fedbn recalib SARTAJ      0.284926      0.283088  0.001838          0.655319    0.004255 606.312500    1.326172
 qafnet     avg SARTAJ      0.284926      0.284926  0.000000          0.659574   -0.004255 606.312500    1.266336
 qafnet recalib SARTAJ      0.297794      0.294118  0.003676          0.663830    0.0000

---
### Reading the output

The claim lives in **`dq_out`** — the FP32→INT8 accuracy drop on the completely unseen site.
QAF-Net should show a markedly smaller drop than FedAvg. `acc_fp32_out` tells you whether
site-local normalization also bought raw cross-site accuracy.

On the CHENG and PMRAM folds every test image is a tumour, so also read **`acc_sub_*`** (subtype
accuracy on the three tumour logits) and **`tumor_missed_int8_out`** (tumours called notumor).

If `dq_out` for FedAvg is already near zero, H1 does not hold for models this small — that is a
legitimate finding and the paper reports it as such. **Do not tune until the hypothesis passes.**

When every `(FOLD, SEED)` combination is done, concatenate the `results.csv` files and hand them
over with the `.tflite` files for the results section.